# 📊 Week 2: Feature Engineering for Time-Series Forecasting

This notebook focuses on preparing a model-ready dataset by creating meaningful features from time-series data.

## 🎯 Objective
- Create date-based features
- Generate lag features
- Build rolling statistics
- Prepare final dataset
- Perform time-series train-test split

## 📥 Data Loading

Load the processed dataset generated from Week 1.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/daily_demand.csv')
df.head()

,date,quantity
0,2024-04-01,475
1,2024-04-02,456
2,2024-04-03,466
3,2024-04-04,415
4,2024-04-05,446


In [5]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      365 non-null    datetime64[ns]
 1   quantity  365 non-null    int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 5.8 KB


## 🗓️ Date Feature Engineering

Convert the date column into datetime format and extract useful features such as day, month, year, and weekday information.

In [7]:
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

df['day_of_week'] = df['date'].dt.dayofweek   # 0 = Monday, 6 = Sunday
df['week_of_year'] = df['date'].dt.isocalendar().week

df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

In [8]:
df.head()

,date,quantity,day,month,year,day_of_week,week_of_year,is_weekend
0,2024-04-01,475,1,4,2024,0,14,0
1,2024-04-02,456,2,4,2024,1,14,0
2,2024-04-03,466,3,4,2024,2,14,0
3,2024-04-04,415,4,4,2024,3,14,0
4,2024-04-05,446,5,4,2024,4,14,0


## ⏳ Lag Features

Lag features help the model understand past patterns by using previous values of the target variable.

In [9]:
df['lag_1'] = df['quantity'].shift(1)
df['lag_7'] = df['quantity'].shift(7)

In [10]:
df.head(10)

,date,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7
0,2024-04-01,475,1,4,2024,0,14,0,NaN,NaN
1,2024-04-02,456,2,4,2024,1,14,0,475.0,NaN
2,2024-04-03,466,3,4,2024,2,14,0,456.0,NaN
3,2024-04-04,415,4,4,2024,3,14,0,466.0,NaN
4,2024-04-05,446,5,4,2024,4,14,0,415.0,NaN
5,2024-04-06,428,6,4,2024,5,14,1,446.0,NaN
6,2024-04-07,417,7,4,2024,6,14,1,428.0,NaN
7,2024-04-08,444,8,4,2024,0,15,0,417.0,475.0
8,2024-04-09,426,9,4,2024,1,15,0,444.0,456.0
9,2024-04-10,474,10,4,2024,2,15,0,426.0,466.0


In [11]:
df['lag_3'] = df['quantity'].shift(3)

In [12]:
df.head(10)

,date,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3
0,2024-04-01,475,1,4,2024,0,14,0,NaN,NaN,NaN
1,2024-04-02,456,2,4,2024,1,14,0,475.0,NaN,NaN
2,2024-04-03,466,3,4,2024,2,14,0,456.0,NaN,NaN
3,2024-04-04,415,4,4,2024,3,14,0,466.0,NaN,475.0
4,2024-04-05,446,5,4,2024,4,14,0,415.0,NaN,456.0
5,2024-04-06,428,6,4,2024,5,14,1,446.0,NaN,466.0
6,2024-04-07,417,7,4,2024,6,14,1,428.0,NaN,415.0
7,2024-04-08,444,8,4,2024,0,15,0,417.0,475.0,446.0
8,2024-04-09,426,9,4,2024,1,15,0,444.0,456.0,428.0
9,2024-04-10,474,10,4,2024,2,15,0,426.0,466.0,417.0


## 📈 Rolling Features

Rolling statistics capture trends and smooth variations in the data over a window of time.

In [13]:
df['rolling_mean_3'] = df['quantity'].rolling(window=3).mean()
df['rolling_mean_7'] = df['quantity'].rolling(window=7).mean()

In [14]:
df.head(10)

,date,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3,rolling_mean_3,rolling_mean_7
0,2024-04-01,475,1,4,2024,0,14,0,NaN,NaN,NaN,NaN,NaN
1,2024-04-02,456,2,4,2024,1,14,0,475.0,NaN,NaN,NaN,NaN
2,2024-04-03,466,3,4,2024,2,14,0,456.0,NaN,NaN,465.666667,NaN
3,2024-04-04,415,4,4,2024,3,14,0,466.0,NaN,475.0,445.666667,NaN
4,2024-04-05,446,5,4,2024,4,14,0,415.0,NaN,456.0,442.333333,NaN
5,2024-04-06,428,6,4,2024,5,14,1,446.0,NaN,466.0,429.666667,NaN
6,2024-04-07,417,7,4,2024,6,14,1,428.0,NaN,415.0,430.333333,443.285714
7,2024-04-08,444,8,4,2024,0,15,0,417.0,475.0,446.0,429.666667,438.857143
8,2024-04-09,426,9,4,2024,1,15,0,444.0,456.0,428.0,429.000000,434.571429
9,2024-04-10,474,10,4,2024,2,15,0,426.0,466.0,417.0,448.000000,435.714286


In [15]:
df['rolling_std_7'] = df['quantity'].rolling(window=7).std()

In [16]:
df.head(10)

,date,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3,rolling_mean_3,rolling_mean_7,rolling_std_7
0,2024-04-01,475,1,4,2024,0,14,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-02,456,2,4,2024,1,14,0,475.0,NaN,NaN,NaN,NaN,NaN
2,2024-04-03,466,3,4,2024,2,14,0,456.0,NaN,NaN,465.666667,NaN,NaN
3,2024-04-04,415,4,4,2024,3,14,0,466.0,NaN,475.0,445.666667,NaN,NaN
4,2024-04-05,446,5,4,2024,4,14,0,415.0,NaN,456.0,442.333333,NaN,NaN
5,2024-04-06,428,6,4,2024,5,14,1,446.0,NaN,466.0,429.666667,NaN,NaN
6,2024-04-07,417,7,4,2024,6,14,1,428.0,NaN,415.0,430.333333,443.285714,23.858711
7,2024-04-08,444,8,4,2024,0,15,0,417.0,475.0,446.0,429.666667,438.857143,19.463030
8,2024-04-09,426,9,4,2024,1,15,0,444.0,456.0,428.0,429.000000,434.571429,18.329004
9,2024-04-10,474,10,4,2024,2,15,0,426.0,466.0,417.0,448.000000,435.714286,20.710016


## 🧹 Handling Missing Values

Lag and rolling operations create missing values at the beginning of the dataset. These rows are removed.

In [17]:
df = df.dropna()

In [18]:
df.head()
df.isnull().sum()

date              0
quantity          0
day               0
month             0
year              0
day_of_week       0
week_of_year      0
is_weekend        0
lag_1             0
lag_7             0
lag_3             0
rolling_mean_3    0
rolling_mean_7    0
rolling_std_7     0
dtype: int64

## 🧱 Final Dataset Preparation

Prepare the dataset for machine learning by removing unnecessary columns like date.

In [19]:
df_model = df.drop(columns=['date'])

In [20]:
df_model.head()

,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3,rolling_mean_3,rolling_mean_7,rolling_std_7
7,444,8,4,2024,0,15,0,417.0,475.0,446.0,429.666667,438.857143,19.463030
8,426,9,4,2024,1,15,0,444.0,456.0,428.0,429.000000,434.571429,18.329004
9,474,10,4,2024,2,15,0,426.0,466.0,417.0,448.000000,435.714286,20.710016
10,449,11,4,2024,3,15,0,474.0,415.0,444.0,449.666667,440.571429,18.954834
11,493,12,4,2024,4,15,0,449.0,446.0,426.0,472.000000,447.285714,27.566370


## 🔀 Train-Test Split (Time-Series)

Split the dataset into training and testing sets based on time order to avoid data leakage.

In [21]:
train_size = int(len(df_model) * 0.8)

train = df_model.iloc[:train_size]
test = df_model.iloc[train_size:]

In [22]:
print(train.shape)
print(test.shape)

(286, 13)
(72, 13)


## 💾 Saving Processed Data

Save the final dataset and train-test splits for future model building.

In [23]:
df_model.to_csv('../data/processed/model_ready_data.csv', index=False)

In [24]:
train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

In [25]:
df.head()

,date,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3,rolling_mean_3,rolling_mean_7,rolling_std_7
7,2024-04-08,444,8,4,2024,0,15,0,417.0,475.0,446.0,429.666667,438.857143,19.463030
8,2024-04-09,426,9,4,2024,1,15,0,444.0,456.0,428.0,429.000000,434.571429,18.329004
9,2024-04-10,474,10,4,2024,2,15,0,426.0,466.0,417.0,448.000000,435.714286,20.710016
10,2024-04-11,449,11,4,2024,3,15,0,474.0,415.0,444.0,449.666667,440.571429,18.954834
11,2024-04-12,493,12,4,2024,4,15,0,449.0,446.0,426.0,472.000000,447.285714,27.566370


In [26]:
train.head()

,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3,rolling_mean_3,rolling_mean_7,rolling_std_7
7,444,8,4,2024,0,15,0,417.0,475.0,446.0,429.666667,438.857143,19.463030
8,426,9,4,2024,1,15,0,444.0,456.0,428.0,429.000000,434.571429,18.329004
9,474,10,4,2024,2,15,0,426.0,466.0,417.0,448.000000,435.714286,20.710016
10,449,11,4,2024,3,15,0,474.0,415.0,444.0,449.666667,440.571429,18.954834
11,493,12,4,2024,4,15,0,449.0,446.0,426.0,472.000000,447.285714,27.566370


In [27]:
test.head()

,quantity,day,month,year,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_3,rolling_mean_3,rolling_mean_7,rolling_std_7
293,463,19,1,2025,6,3,1,529.0,460.0,463.0,472.666667,466.857143,30.748597
294,460,20,1,2025,0,4,0,463.0,467.0,426.0,484.000000,465.857143,30.856812
295,463,21,1,2025,1,4,0,460.0,463.0,529.0,462.000000,465.857143,30.856812
296,406,22,1,2025,2,4,0,463.0,457.0,463.0,443.000000,458.571429,38.396428
297,404,23,1,2025,3,4,0,406.0,463.0,460.0,424.333333,450.142857,43.410554


In [28]:
print("Full Data:", df_model.shape)
print("Train:", train.shape)
print("Test:", test.shape)

Full Data: (358, 13)
Train: (286, 13)
Test: (72, 13)


## ✅ Summary

- Created date-based features
- Generated lag features (lag_1, lag_3, lag_7)
- Computed rolling statistics
- Handled missing values
- Prepared model-ready dataset
- Applied time-series train-test split
- Saved processed datasets

🚀 The dataset is now ready for model building.